# Sales bookings and fulfillment

Inspect the order-line and shipment-line grains and the shared dimensions. The two baseline orders should produce two booking rows, 11 pumps, and ₹140,000.

Run from the `gold/` directory after setting `silver_manifest` in `gold_settings.json`.

In [ ]:
import json
from pathlib import Path

from IPython.display import display
import pandas as pd

import gold_pipeline
from gold_pipeline import build, publish

SETTINGS_PATH = Path(gold_pipeline.__file__).resolve().with_name("gold_settings.json")
settings_snapshot = SETTINGS_PATH.read_bytes()
configured_manifest = json.loads(settings_snapshot.decode("utf-8-sig")).get("silver_manifest")
print("Gold settings:", SETTINGS_PATH)
print("Silver manifest:", configured_manifest)
gold = build(SETTINGS_PATH, stage="bookings")
print("Silver release", gold.manifest["model_run_id"])
print("Cutoff", gold.settings["cutoff"])
display(pd.DataFrame([{"table": name, "rows": len(rows)} for name, rows in gold.tables.items()]))

In [ ]:
baseline = {"P-S1", "D-S1"}
display(pd.DataFrame([
    row for row in gold.records("fact_sales_order_booking")
    if row["order_reference"] in baseline
]))

In [ ]:
for name in ("dim_customer", "dim_product", "dim_legal_entity", "fact_shipment_line"):
    print(name)
    display(pd.DataFrame(gold.records(name)).head(20))

In [ ]:
display(pd.DataFrame(gold.checks))
display(pd.DataFrame(gold.exceptions))